# Case study: a data quality report for the review data

*Session 4, block 1 · Author: course team · Licence: CC-BY-4.0*

**Task.** Write a data quality report for the review data: which checks did you run, what did they find,
how severe is each finding, and what should happen next?

The notebook expresses every check as code, so that the report can be regenerated after each data update.
Section 4 writes the same rules as a **pandera** schema, and the file
[`quality/test_review_quality.py`](quality/test_review_quality.py) contains them as **pytest** tests.

Cells marked **TODO** are for you.

In [ ]:
from pathlib import Path

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
DATA = ROOT / "case-study" / "data"
assert (DATA / "train.parquet").exists(), "run case-study/prepare_data.py first"

import numpy as np
import pandas as pd

reviews = pd.read_parquet(DATA / "train.parquet")
products = pd.read_parquet(DATA / "products.parquet")
print(reviews.shape, products.shape)

## 1. A small framework: every check returns a row

Each check records the **dimension** of data quality it belongs to, the table and column, the number of
failing rows and a **severity**:

- *critical*: breaks a key or a hard rule; the data cannot be used as they are;
- *major*: affects many rows or a central variable; must be handled before analysis;
- *minor*: worth documenting; handle when it matters for a question.

In [ ]:
results = []


def check(dimension, table, column, description, failed, n_total, severity):
    """Record one check. `failed` is a boolean Series (True = row fails) or a count."""
    n_failed = int(failed.sum()) if hasattr(failed, "sum") else int(failed)
    results.append({"dimension": dimension, "table": table, "column": column, "check": description,
                    "n_failed": n_failed, "share": n_failed / n_total,
                    "severity": severity if n_failed else "ok"})

## 2. Checks on the reviews

### Completeness: missing values and hidden missing values

A value can be missing as `NULL`/`NaN` or *hidden* as an empty string, a placeholder ("n/a", "-") or a
default value. Both count as incomplete.

In [ ]:
n = len(reviews)
for col in reviews.columns:
    check("completeness", "reviews", col, "value is NULL", reviews[col].isna(), n, "major")
check("completeness", "reviews", "text", "text is empty or only white space",
      reviews["text"].str.strip().eq(""), n, "major")
check("completeness", "reviews", "text", "text shorter than 3 characters",
      reviews["text"].str.strip().str.len().lt(3), n, "minor")

### Validity: types and ranges

In [ ]:
expected_types = {"review_id": "str", "rating": "int64", "helpful_vote": "int64",
                  "verified_purchase": "bool", "date": "datetime64[ms]", "n_images": "int64", "label": "str"}
for col, dtype in expected_types.items():
    check("validity", "reviews", col, f"dtype is {dtype}", int(str(reviews[col].dtype) != dtype), 1, "critical")

check("validity", "reviews", "rating", "rating in 1..5", ~reviews["rating"].between(1, 5), n, "critical")
check("validity", "reviews", "label", "label in {neg, neu, pos}", ~reviews["label"].isin(["neg", "neu", "pos"]), n, "critical")
check("validity", "reviews", "helpful_vote", "helpful_vote >= 0", reviews["helpful_vote"].lt(0), n, "critical")
check("validity", "reviews", "n_images", "n_images >= 0", reviews["n_images"].lt(0), n, "critical")
check("validity", "reviews", "date", "date between 2001-01-01 and 2021-12-31",
      ~reviews["date"].between("2001-01-01", "2021-12-31 23:59:59"), n, "critical")

### Uniqueness: keys and duplicates

In [ ]:
check("uniqueness", "reviews", "review_id", "review_id unique", reviews["review_id"].duplicated(), n, "critical")
check("uniqueness", "reviews", "user_id, parent_asin", "one review per user and product",
      reviews.duplicated(["user_id", "parent_asin"]), n, "major")
check("uniqueness", "reviews", "text", "text appears more than once (exact duplicate)",
      reviews["text"].duplicated(), n, "minor")
reviews["text"].value_counts().head(5)

### Consistency: rules that relate columns and tables

In [ ]:
expected_label = pd.cut(reviews["rating"], bins=[0, 2, 3, 5], labels=["neg", "neu", "pos"]).astype(str)
check("consistency", "reviews", "label", "label matches rating (1-2 neg, 3 neu, 4-5 pos)",
      reviews["label"].ne(expected_label), n, "critical")
check("consistency", "reviews", "parent_asin", "product exists in products (foreign key)",
      ~reviews["parent_asin"].isin(products["parent_asin"]), n, "critical")

recomputed = reviews.groupby("parent_asin")["rating"].agg(avg="mean", n="size")
joined = products.set_index("parent_asin").join(recomputed, how="inner")
check("consistency", "products", "train_avg_rating", "equals the mean rating of its reviews",
      ~np.isclose(joined["train_avg_rating"], joined["avg"]), len(joined), "critical")
check("consistency", "products", "train_n_reviews", "equals the number of its reviews",
      joined["train_n_reviews"].ne(joined["n"]), len(joined), "critical")

### Plausibility: things that are valid but suspicious

In [ ]:
check("plausibility", "reviews", "text", "contains HTML line breaks (<br />)",
      reviews["text"].str.contains("<br", regex=False), n, "minor")
check("plausibility", "reviews", "text", "contains a no-break space (U+00A0)",
      reviews["text"].str.contains("\u00a0", regex=False), n, "minor")
check("plausibility", "reviews", "helpful_vote", "helpful_vote above 1000",
      reviews["helpful_vote"].gt(1000), n, "minor")

## 3. Checks on the products

In [ ]:
m = len(products)
check("uniqueness", "products", "parent_asin", "parent_asin unique", products["parent_asin"].duplicated(), m, "critical")
for col in ["price", "store", "details", "train_avg_rating"]:
    check("completeness", "products", col, "value is NULL", products[col].isna(), m, "major")
for col in ["title", "features", "description", "categories"]:
    check("completeness", "products", col, "empty string (hidden missing)", products[col].eq(""), m, "major")
check("validity", "products", "price", "price > 0", products["price"].le(0), m, "critical")
check("plausibility", "products", "price", "price above 1000", products["price"].gt(1000), m, "minor")
check("validity", "products", "main_category", "only one category", int(products["main_category"].nunique() != 1), 1, "minor")

## 4. The report

In [ ]:
order = {"critical": 0, "major": 1, "minor": 2, "ok": 3}
report = (pd.DataFrame(results)
          .assign(rank=lambda d: d["severity"].map(order), share=lambda d: d["share"].round(4))
          .sort_values(["rank", "n_failed"], ascending=[True, False])
          .drop(columns="rank")
          .reset_index(drop=True))
pd.set_option("display.max_colwidth", 60)
report[report["severity"] != "ok"]

In [ ]:
print(f"{len(report)} checks, {(report['severity'] == 'ok').sum()} passed")
report.groupby(["dimension", "severity"]).size().unstack(fill_value=0)

In [ ]:
def to_markdown(df: pd.DataFrame) -> str:
    """A small Markdown table writer (avoids an extra dependency)."""
    lines = ["| " + " | ".join(df.columns) + " |", "|" + "---|" * len(df.columns)]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)


findings = report[report["severity"] != "ok"][["severity", "dimension", "table", "column", "check", "n_failed", "share"]]
text = f"""# Data quality report: review data

Data: `train.parquet` ({len(reviews):,} reviews) and `products.parquet` ({len(products):,} products).
Checks run: {len(report)}; passed: {(report['severity'] == 'ok').sum()}.

## Findings

{to_markdown(findings)}

## Recommended actions

1. (TODO) ...
"""
print(text[:2500])

**TODO 1.** Write the *Recommended actions*: for each major finding, say what should be done (remove,
correct, flag, impute, document) and why. Example: "Products: price missing for 82.5 %. Do not drop these
products; analyse whether missingness depends on popularity (block 2) and add a missing-value indicator."

**TODO 2.** Add two checks of your own, for example: is `title` identical to `text` for some reviews? Do
some users write implausibly many reviews on one day?

In [ ]:
# TODO 2

## 5. The same rules as a pandera schema

[pandera](https://pandera.readthedocs.io/) (MIT licence) describes the expected structure of a DataFrame as a
**schema**: columns, types and checks. `validate(lazy=True)` runs all checks and collects every failure
instead of stopping at the first one. Workbook 01 introduces pandera in general.

Install it for this notebook with `uv run --with pandera jupyter lab`.

In [ ]:
try:
    import pandera.pandas as pa
except ImportError:
    pa = None
    print("pandera is not installed: start Jupyter with  uv run --with pandera jupyter lab")

if pa is not None:
    review_schema = pa.DataFrameSchema(
        {
            "review_id": pa.Column(str, unique=True),
            "rating": pa.Column(int, pa.Check.in_range(1, 5)),
            "text": pa.Column(str, pa.Check(lambda s: s.str.strip().str.len() > 0, error="text is empty")),
            "parent_asin": pa.Column(str, pa.Check.isin(products["parent_asin"])),
            "helpful_vote": pa.Column(int, pa.Check.ge(0)),
            "verified_purchase": pa.Column(bool),
            "date": pa.Column("datetime64[ms]", pa.Check.in_range(pd.Timestamp("2001-01-01"), pd.Timestamp("2022-01-01"))),
            "label": pa.Column(str, pa.Check.isin(["neg", "neu", "pos"])),
        },
        unique=["user_id", "parent_asin"],     # one review per user and product
        strict=False,                          # other columns are allowed
    )
    try:
        review_schema.validate(reviews, lazy=True)
        print("all checks passed")
    except pa.errors.SchemaErrors as err:
        failures = err.failure_cases
        print(failures.groupby(["column", "check"], dropna=False).size())

## 6. Validation rules as tests

A **test** is a function that fails loudly when a rule is broken. Running the tests after every data update
(locally or in continuous integration, Session 2) turns the quality report into a guard. From the repository
root:

```bash
uv run --with pytest pytest sessions/04-data-quality/workbooks/quality -v
```

Known problems of the raw data are marked as *expected failures* (`xfail`) with the reason, so that they are
documented without making the test run fail. After block 3, the tests on the cleaned table must pass.

In [ ]:
print((Path("quality") / "test_review_quality.py").read_text()[:2000] if Path("quality").exists() else
      (ROOT / "sessions/04-data-quality/workbooks/quality/test_review_quality.py").read_text()[:2000])